In [ ]:
import json, random
from google.colab import drive, userdata
from datasets import load_dataset

drive.mount('/content/drive')

with open('/content/drive/MyDrive/session_final.json') as f:
    prev_session = json.load(f)
print("Loaded prior session findings.")
print(f"Primary finding to replicate: {prev_session['primary_findings_robust']['unanimous_wrong_convergence']}")

print("\nLoading MedMCQA...")
medmcqa = load_dataset("openlifescienceai/medmcqa")
print(f"\nSplits: {list(medmcqa.keys())}")
for split in medmcqa.keys():
    print(f"  {split}: {len(medmcqa[split])} questions")

print(f"\nSchema (first row keys): {list(medmcqa['train'][0].keys())}")
print(f"\nFirst row sample (truncated):")
for k, v in medmcqa['train'][0].items():
    print(f"  {k}: {str(v)[:120]}")

print(f"\nAnswer field check (first 5 questions):")
for i in range(5):
    row = medmcqa['train'][i]

    for field in ['cop', 'answer', 'correct_option', 'label', 'answer_idx']:
        if field in row:
            print(f"  Q{i}: {field}={row[field]}")
            break

In [ ]:
import json
from collections import Counter

val = medmcqa['validation']
single_choice_idx = sorted([i for i, row in enumerate(val) if row['choice_type'] == 'single'])
print(f"Single-choice validation questions: {len(single_choice_idx)}")

LETTER_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
formatted_questions = []
for i in single_choice_idx:
    row = val[i]
    formatted_questions.append({
        'idx': i,
        'id': row['id'],
        'question': row['question'],
        'options': {
            'A': row['opa'],
            'B': row['opb'],
            'C': row['opc'],
            'D': row['opd'],
        },
        'answer_idx': LETTER_MAP[row['cop']],
        'subject': row['subject_name'],
        'topic': row['topic_name'],
    })

clean = [q for q in formatted_questions
         if all(q['options'][L] for L in 'ABCD')
         and q['answer_idx'] in 'ABCD'
         and q['question']]
dropped = len(formatted_questions) - len(clean)
print(f"After dropping malformed rows: {len(clean)} (dropped {dropped})")

formatted_questions = clean

subjects = Counter(q['subject'] for q in formatted_questions)
print(f"\nFinal subject distribution:")
for subj, count in subjects.most_common():
    pct = count / len(formatted_questions) * 100
    print(f"  {subj}: {count} ({pct:.1f}%)")

with open('/content/drive/MyDrive/medmcqa_sample.json', 'w') as f:
    json.dump(formatted_questions, f)
print(f"\nSaved /content/drive/MyDrive/medmcqa_sample.json")
print(f"Total questions: {len(formatted_questions)}")

print(f"\nFirst question:")
q = formatted_questions[0]
print(f"  ID: {q['id']}")
print(f"  Subject: {q['subject']} / {q['topic']}")
print(f"  Question: {q['question'][:200]}{'...' if len(q['question'])>200 else ''}")
for letter, text in q['options'].items():
    marker = " ← GOLD" if letter == q['answer_idx'] else ""
    print(f"    {letter}: {text}{marker}")

N = len(formatted_questions)
scale = N / 1273
print(f"\n=== Cost estimate for {N} questions ({scale:.1f}× MedQA size) ===")
print(f"  Llama-3-8B-Lite:    ~${0.20 * scale:.2f}")
print(f"  Qwen-2.5-7B-Turbo:  ~${0.25 * scale:.2f}")
print(f"  Gemma-3n-E4B:       ~${0.15 * scale:.2f}")
print(f"  DeepSeek-V3:        ~${0.60 * scale:.2f}")
print(f"  Flan-t5-base:       FREE (local CPU)")
print(f"  TOTAL Together:     ~${(0.20 + 0.25 + 0.15 + 0.60) * scale:.2f}")
print(f"\n  Your current balance: ~$4 — sufficient with buffer")

In [ ]:
import json
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("Loading Flan-t5-base...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
model.eval()

with open('/content/drive/MyDrive/medmcqa_sample.json') as f:
    questions = json.load(f)
print(f"Loaded {len(questions)} questions")

def make_prompt(q):
    return f"""Answer this medical question. Reply with only A, B, C, or D.

Question: {q['question']}

A: {q['options']['A']}
B: {q['options']['B']}
C: {q['options']['C']}
D: {q['options']['D']}

Answer:"""

results = []
for i, q in enumerate(questions):
    prompt = make_prompt(q)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True).strip().upper()
    pred = next((c for c in text if c in 'ABCD'), None)

    results.append({
        'idx': q['idx'],
        'id': q['id'],
        'gold': q['answer_idx'],
        'pred': pred,
        'raw_output': text,
        'correct': int(pred == q['answer_idx']) if pred else 0,
    })

    if (i + 1) % 200 == 0:
        acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
        print(f"  {i+1}/{len(questions)} — running acc: {acc_so_far:.1f}%")

total = len(results)
correct = sum(r['correct'] for r in results)
acc = correct / total
print(f"\n=== Flan-t5-base on MedMCQA ===")
print(f"  Accuracy: {correct}/{total} = {acc*100:.1f}%")
print(f"  (For comparison: Flan-t5 on MedQA-USMLE was 26.5%)")

with open('/content/drive/MyDrive/medmcqa_flan_t5_results.json', 'w') as f:
    json.dump(results, f)
print(f"  Saved to /content/drive/MyDrive/medmcqa_flan_t5_results.json")

from collections import Counter
pred_dist = Counter(r['pred'] for r in results)
print(f"\nPrediction distribution: {dict(pred_dist)}")

In [ ]:
import requests, json, time

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

with open('/content/drive/MyDrive/medmcqa_sample.json') as f:
    questions = json.load(f)

def get_answer(question, options, api_key, model):
    prompt = f"""Answer this medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 5,
                "temperature": 0.0
            },
            timeout=30
        )
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}: {r.text[:150]}"
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in 'ABCD':
                return char, None
        return None, f"No letter: {text!r}"
    except Exception as e:
        return None, str(e)

MODEL = "meta-llama/Meta-Llama-3-8B-Instruct-Lite"
print(f"Running {MODEL} on {len(questions)} MedMCQA questions...")

results = []
consecutive_errors = 0
for i, q in enumerate(questions):
    pred, err = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)

    if pred is None:
        consecutive_errors += 1
        if consecutive_errors <= 3:
            print(f"  [{i}] error: {err}")
        if consecutive_errors >= 10:
            print(f"  >>> 10 consecutive errors, aborting <<<")
            break
    else:
        consecutive_errors = 0

    results.append({
        'idx': q['idx'],
        'id': q['id'],
        'gold': q['answer_idx'],
        'pred': pred,
        'correct': int(pred == q['answer_idx']) if pred else 0,
    })

    if (i + 1) % 200 == 0:
        acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
        print(f"  {i+1}/{len(questions)} — running acc: {acc_so_far:.1f}%")
    time.sleep(0.1)

total = len(results)
correct = sum(r['correct'] for r in results)
print(f"\n=== Llama-3-8B-Lite on MedMCQA ===")
print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"  (For comparison: Llama on MedQA-USMLE was 52.0%)")

from collections import Counter
pred_dist = Counter(r['pred'] for r in results)
print(f"  Prediction distribution: {dict(pred_dist)}")

with open('/content/drive/MyDrive/medmcqa_llama_results.json', 'w') as f:
    json.dump(results, f)
print(f"  Saved to /content/drive/MyDrive/medmcqa_llama_results.json")

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct-Turbo"
print(f"Running {MODEL} on {len(questions)} MedMCQA questions...")

results = []
consecutive_errors = 0
for i, q in enumerate(questions):
    pred, err = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)

    if pred is None:
        consecutive_errors += 1
        if consecutive_errors <= 3:
            print(f"  [{i}] error: {err}")
        if consecutive_errors >= 10:
            print(f"  >>> 10 consecutive errors, aborting <<<")
            break
    else:
        consecutive_errors = 0

    results.append({
        'idx': q['idx'],
        'id': q['id'],
        'gold': q['answer_idx'],
        'pred': pred,
        'correct': int(pred == q['answer_idx']) if pred else 0,
    })

    if (i + 1) % 200 == 0:
        acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
        print(f"  {i+1}/{len(questions)} — running acc: {acc_so_far:.1f}%")
    time.sleep(0.1)

total = len(results)
correct = sum(r['correct'] for r in results)
print(f"\n=== Qwen-2.5-7B on MedMCQA ===")
print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"  (For comparison: Qwen on MedQA-USMLE was 59.5%)")

from collections import Counter
pred_dist = Counter(r['pred'] for r in results)
print(f"  Prediction distribution: {dict(pred_dist)}")

with open('/content/drive/MyDrive/medmcqa_qwen_results.json', 'w') as f:
    json.dump(results, f)
print(f"  Saved to /content/drive/MyDrive/medmcqa_qwen_results.json")

In [ ]:
MODEL = "google/gemma-3n-E4B-it"
print(f"Running {MODEL} on {len(questions)} MedMCQA questions...")

results = []
consecutive_errors = 0
for i, q in enumerate(questions):
    pred, err = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)

    if pred is None:
        consecutive_errors += 1
        if consecutive_errors <= 3:
            print(f"  [{i}] error: {err}")
        if consecutive_errors >= 10:
            print(f"  >>> 10 consecutive errors, aborting <<<")
            break
    else:
        consecutive_errors = 0

    results.append({
        'idx': q['idx'],
        'id': q['id'],
        'gold': q['answer_idx'],
        'pred': pred,
        'correct': int(pred == q['answer_idx']) if pred else 0,
    })

    if (i + 1) % 200 == 0:
        acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
        print(f"  {i+1}/{len(questions)} — running acc: {acc_so_far:.1f}%")
    time.sleep(0.1)

total = len(results)
correct = sum(r['correct'] for r in results)
print(f"\n=== Gemma-3n-E4B on MedMCQA ===")
print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"  (For comparison: Gemma-3n on MedQA-USMLE was 53.9%)")

from collections import Counter
pred_dist = Counter(r['pred'] for r in results)
print(f"  Prediction distribution: {dict(pred_dist)}")

with open('/content/drive/MyDrive/medmcqa_gemma3n_results.json', 'w') as f:
    json.dump(results, f)
print(f"  Saved to /content/drive/MyDrive/medmcqa_gemma3n_results.json")

In [ ]:
MODEL = "deepseek-ai/DeepSeek-V3"
print(f"Running {MODEL} on {len(questions)} MedMCQA questions...")
print("This is the slow one — go get coffee, but check back in 30 min\n")

results = []
consecutive_errors = 0
for i, q in enumerate(questions):
    pred, err = get_answer(q['question'], q['options'], TOGETHER_API_KEY, MODEL)

    if pred is None:
        consecutive_errors += 1
        if consecutive_errors <= 3:
            print(f"  [{i}] error: {err}")
        if consecutive_errors >= 10:
            print(f"  >>> 10 consecutive errors, aborting <<<")
            break
    else:
        consecutive_errors = 0

    results.append({
        'idx': q['idx'],
        'id': q['id'],
        'gold': q['answer_idx'],
        'pred': pred,
        'correct': int(pred == q['answer_idx']) if pred else 0,
    })

    if (i + 1) % 200 == 0:
        acc_so_far = sum(r['correct'] for r in results) / len(results) * 100

        with open('/content/drive/MyDrive/medmcqa_deepseek_results_partial.json', 'w') as f:
            json.dump(results, f)
        print(f"  {i+1}/{len(questions)} — running acc: {acc_so_far:.1f}%  (checkpoint saved)")
    time.sleep(0.15)

total = len(results)
correct = sum(r['correct'] for r in results)
print(f"\n=== DeepSeek-V3 on MedMCQA ===")
print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"  (For comparison: DeepSeek-V3 on MedQA-USMLE was 77.7%)")

from collections import Counter
pred_dist = Counter(r['pred'] for r in results)
print(f"  Prediction distribution: {dict(pred_dist)}")

with open('/content/drive/MyDrive/medmcqa_deepseek_results.json', 'w') as f:
    json.dump(results, f)
print(f"  Saved to /content/drive/MyDrive/medmcqa_deepseek_results.json")

In [ ]:
r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
    json={
        "model": "deepseek-ai/DeepSeek-V3",
        "messages": [{"role": "user", "content": "Reply PING"}],
        "max_tokens": 5,
        "temperature": 0.0
    },
    timeout=20
)
print(f"Status: {r.status_code}")
print(f"Response: {r.text[:300]}")

In [ ]:
import json, numpy as np
from collections import Counter

def load_results(path):
    with open(path) as f:
        return {r['id']: r for r in json.load(f)}

flan = load_results('/content/drive/MyDrive/medmcqa_flan_t5_results.json')
llama = load_results('/content/drive/MyDrive/medmcqa_llama_results.json')
qwen = load_results('/content/drive/MyDrive/medmcqa_qwen_results.json')
gemma = load_results('/content/drive/MyDrive/medmcqa_gemma3n_results.json')

with open('/content/drive/MyDrive/medmcqa_sample.json') as f:
    questions = json.load(f)

ids = [q['id'] for q in questions]
N = len(ids)

def correct_vec(d, ids):
    return np.array([d[i]['correct'] if i in d and d[i].get('pred') is not None else 0 for i in ids])

f_c = correct_vec(flan, ids)
l_c = correct_vec(llama, ids)
q_c = correct_vec(qwen, ids)
g_c = correct_vec(gemma, ids)

print(f"=== MedMCQA 4-model results (n={N}) ===")
print(f"  Flan-t5-base:    {f_c.mean()*100:.1f}%")
print(f"  Llama-3-8B-Lite: {l_c.mean()*100:.1f}%")
print(f"  Qwen-2.5-7B:     {q_c.mean()*100:.1f}%")
print(f"  Gemma-3n-E4B:    {g_c.mean()*100:.1f}%")

print(f"\n=== k-intersection: Flan right AND k of 3 strong models wrong ===\n")
print(f"{'k':<6} {'Observed':<12} {'Null mean':<14} {'Null 95% CI':<18} {'Z':<8}")
print("-" * 60)

B = 10000
rng = np.random.default_rng(seed=42)

for k in range(4):
    wrong_count = (l_c == 0).astype(int) + (q_c == 0).astype(int) + (g_c == 0).astype(int)
    observed = int(((f_c == 1) & (wrong_count == k)).sum())
    null = np.zeros(B, dtype=int)
    for b in range(B):
        f_s = rng.permutation(f_c)
        l_s = rng.permutation(l_c)
        q_s = rng.permutation(q_c)
        g_s = rng.permutation(g_c)
        wc = (l_s == 0).astype(int) + (q_s == 0).astype(int) + (g_s == 0).astype(int)
        null[b] = int(((f_s == 1) & (wc == k)).sum())
    nm, ns = null.mean(), null.std()
    lo, hi = np.percentile(null, [2.5, 97.5])
    z = (observed - nm) / ns if ns > 0 else 0
    print(f"k={k}   {observed:<12} {nm:<14.1f} [{lo:.0f}, {hi:.0f}]{'':<6} {z:+.2f}")

all_strong_wrong = (l_c == 0) & (q_c == 0) & (g_c == 0)
strict_trap_count = int(((f_c == 1) & all_strong_wrong).sum())
all_strong_wrong_count = int(all_strong_wrong.sum())

print(f"\n=== All-strong-wrong subset (3 strong models) ===")
print(f"  Total all-strong-wrong: {all_strong_wrong_count}")
print(f"  Flan right within subset: {strict_trap_count} ({strict_trap_count/max(1,all_strong_wrong_count)*100:.1f}%)")
print(f"  (Flan marginal: {f_c.mean()*100:.1f}%)")

print(f"\n=== Wrong-answer convergence on all-strong-wrong subset ===")
all_wrong_ids = [ids[i] for i in range(N) if all_strong_wrong[i]]
unanimous_count = 0
agreement_dist = Counter()
for qid in all_wrong_ids:
    preds = [llama[qid]['pred'], qwen[qid]['pred'], gemma[qid]['pred']]
    if None in preds:
        continue
    pc = Counter(preds)
    most_common_count = pc.most_common(1)[0][1]
    agreement_dist[most_common_count] += 1
    if most_common_count == 3:
        unanimous_count += 1

print(f"  Agreement distribution (3 strong models):")
for k in sorted(agreement_dist.keys(), reverse=True):
    n = agreement_dist[k]
    pct = n / max(1, sum(agreement_dist.values())) * 100
    label = "UNANIMOUS" if k == 3 else f"{k}/3 agree"
    print(f"    {label}: {n} ({pct:.1f}%)")

chance_unanim = (1/3)**2
total_classified = max(1, sum(agreement_dist.values()))
observed_unanim_rate = unanimous_count / total_classified
print(f"\n  Chance rate of 3-way unanimous (independent picks among 3 distractors): {chance_unanim*100:.1f}%")
print(f"  Observed unanimous rate: {observed_unanim_rate*100:.1f}%")
print(f"  Ratio to chance: {observed_unanim_rate/chance_unanim:.1f}x")

print(f"\n=== Pairwise failure lift ===\n")
names = {'flan': f_c, 'llama': l_c, 'qwen': q_c, 'gemma': g_c}
ordered = list(names.keys())
for i, n1 in enumerate(ordered):
    for n2 in ordered[i+1:]:
        a = names[n1]; b = names[n2]
        a_wrong = (a == 0).sum()
        p_b_wrong = (b == 0).mean()
        if a_wrong == 0 or p_b_wrong == 0:
            continue
        both_wrong = ((a == 0) & (b == 0)).sum()
        cond_b_given_a = both_wrong / a_wrong
        lift = cond_b_given_a / p_b_wrong
        print(f"  {n1:6s} ↔ {n2:6s}: lift = {lift:.2f}x")

print(f"\n=== Comparison to MedQA findings (full 4-strong model version) ===")
print(f"  MedQA k=4 all-strong-wrong: z = +9.61 (n=1273)")
print(f"  MedQA unanimous-wrong: 33.6% vs 3.7% chance (9.1x)")
print(f"  MedQA strong-strong lifts: 1.39-1.74x")
print(f"  MedQA Flan-strong lifts: 1.02-1.07x")
print(f"\n  NOTE: This MedMCQA analysis uses only 3 strong models (no DeepSeek yet).")
print(f"  Chance rates differ between 3-strong and 4-strong setups; the question is")
print(f"  whether the OBSERVED-vs-NULL excess pattern replicates direction and magnitude.")

In [ ]:
import json
from datetime import datetime

medmcqa_summary = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'MedMCQA validation (single-choice, n=2816)',
    'note': 'DeepSeek-V3 pending due to Together 503',

    'model_accuracies': {
        'flan_t5_base': 0.268,
        'llama_3_8b_lite': 0.546,
        'qwen_25_7b_turbo': 0.562,
        'gemma_3n_e4b': 0.492,
    },

    'k_intersection_z': {
        'k=0': 13.60,
        'k=1': -9.01,
        'k=2': -5.56,
        'k=3': 8.36,
    },

    'all_strong_wrong': {
        'n_subset': 623,
        'flan_correct_rate': 0.226,
        'flan_marginal_rate': 0.268,
        'expertise_reversal_replicated_null': True,
    },

    'unanimous_wrong': {
        'n_unanimous': 227,
        'n_classified': 623,
        'observed_rate': 0.364,
        'chance_rate_3models': 0.111,
        'ratio_to_chance': 3.3,
        'medqa_observed_rate': 0.336,
        'note': 'Raw rate nearly identical to MedQA (33.6% vs 36.4%)'
    },

    'pairwise_lifts': {
        'flan_llama': 1.03, 'flan_qwen': 1.01, 'flan_gemma': 1.03,
        'llama_qwen': 1.40, 'llama_gemma': 1.33, 'qwen_gemma': 1.48,
    },

    'replication_status': {
        'shared_blindspots': 'REPLICATED (lifts within 0.05 of MedQA)',
        'unanimous_convergence': 'REPLICATED (raw rate matches MedQA)',
        'flan_independence': 'REPLICATED (lifts ~1.0)',
        'expertise_reversal_null': 'REPLICATED (Flan no better on shared-fail subset)',
    },

    'tomorrow_priority': 'Retry DeepSeek-V3 on MedMCQA when Together recovers; will confirm cross-tier story'
}

with open('/content/drive/MyDrive/medmcqa_summary.json', 'w') as f:
    json.dump(medmcqa_summary, f, indent=2)

print("Saved /content/drive/MyDrive/medmcqa_summary.json")
print(f"\nReplication status:")
for finding, status in medmcqa_summary['replication_status'].items():
    print(f"  {finding}: {status}")
print(f"\nPaper now has two-dataset replication. DeepSeek confirmatory, not essential.")

In [ ]:
import os, shutil

if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("Cleared /content/drive (local residue)")
else:
    print("/content/drive does not exist — good")

from google.colab import drive
drive.mount('/content/drive')

files = os.listdir('/content/drive/MyDrive')
project_files = [f for f in files if any(kw in f for kw in
    ['medmcqa', 'bias', 'session', 'flan_t5', 'llama_results', 'qwen_results',
     'gemma3n', 'deepseek', 'null_model', 'non_trap', 'trap_expl',
     'docs.pkl', 'pubmedqa', 'doc_embeddings', 'tonight'])]
print(f"\nDrive mounted. Project files visible in root: {len(project_files)}")
for f in sorted(project_files):
    print(f"  {f}")

In [ ]:
import os, shutil
from datetime import datetime

PROJECT_DIR = '/content/drive/MyDrive'
SOURCE_DIR = '/content/drive/MyDrive'

os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Target folder: {PROJECT_DIR}\n")

PROJECT_FILES = [

    'flan_t5_results.json', 'llama_results.json', 'qwen_results.json',
    'gemma3n_results.json', 'deepseek_results.json',

    'medmcqa_sample.json', 'medmcqa_flan_t5_results.json',
    'medmcqa_llama_results.json', 'medmcqa_qwen_results.json',
    'medmcqa_gemma3n_results.json', 'medmcqa_deepseek_results.json',
    'medmcqa_deepseek_results_partial.json',

    'bias_labels.json', 'non_trap_bias_labels.json',
    'shared_failure_bias_labels.json', 'trap_explanations.json',

    'bias_labels_gpt4omini.json', 'shared_failure_bias_gpt4omini.json',

    'non_trap_sample_idx.json',

    'null_model_results.json', 'tonight_summary.json',
    'session_2_summary.json', 'session_final.json', 'medmcqa_summary.json',

    'docs.pkl', 'pubmedqa_index.faiss', 'doc_embeddings.npy',
]

moved, missing, already = [], [], []
for fname in PROJECT_FILES:
    src = os.path.join(SOURCE_DIR, fname)
    dst = os.path.join(PROJECT_DIR, fname)
    if os.path.exists(dst):
        already.append(fname)
        continue
    if os.path.exists(src):
        shutil.move(src, dst)
        moved.append((fname, os.path.getsize(dst)))
    else:
        missing.append(fname)

print(f"=== MOVED: {len(moved)} files ===")
for fname, size in sorted(moved):
    s = f"{size/1024/1024:.1f} MB" if size > 1024*1024 else f"{size/1024:.1f} KB"
    print(f"  ✓ {fname:<45s} {s:>10s}")

if already:
    print(f"\n=== Already in /: {len(already)} ===")
    for f in already: print(f"  • {f}")
if missing:
    print(f"\n=== Not found (skipped): {len(missing)} ===")
    for f in missing: print(f"  - {f}")

print(f"\n=== CONTENTS OF / ===")
total = 0
for fname in sorted(os.listdir(PROJECT_DIR)):
    full = os.path.join(PROJECT_DIR, fname)
    if os.path.isfile(full):
        size = os.path.getsize(full)
        total += size
        s = f"{size/1024/1024:.1f} MB" if size > 1024*1024 else f"{size/1024:.1f} KB"
        print(f"  {fname:<45s} {s:>10s}")
print(f"\nTotal: {len(os.listdir(PROJECT_DIR))} items, {total/1024/1024:.1f} MB")

readme = f"""ICDM 2026 — Convergent Failure Modes in Clinical LLM Reasoning
Last organized: {datetime.now().strftime('%Y-%m-%d %H:%M')}

DATASETS: MedQA-USMLE (1,273) | MedMCQA validation single-choice (2,816)
MODELS: Flan-t5-base, Llama-3-8B-Lite, Qwen-2.5-7B, Gemma-3n-E4B, DeepSeek-V3
CLASSIFIERS: Llama-3.3-70B-Instruct-Turbo (primary), GPT-4o-mini (triangulation)

PRIMARY FINDING — replicated across both datasets:
  Unanimous-wrong distractor convergence across LLM families.
  MedQA:   33.6% observed vs 3.7% chance (9.1x; 4-strong models)
  MedMCQA: 36.4% observed vs 11.1% chance (3.3x; 3-strong models; DeepSeek pending)
"""
with open(os.path.join(PROJECT_DIR, 'README.md'), 'w') as f:
    f.write(readme)
print(f"\nWrote README.md")

In [ ]:
import json, requests, time, os
from collections import Counter
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    questions = json.load(f)

print(f"Project dir: {PROJECT_DIR}")
print(f"Together key loaded: {bool(TOGETHER_API_KEY)}")
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)}")
print(f"MedMCQA questions loaded: {len(questions)}")
print(f"(Should be 2816)")

In [ ]:
import json, requests, time
from collections import Counter
from google.colab import userdata

PROJECT_DIR = '/content/drive/MyDrive'
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    questions = json.load(f)

print("=== Test call ===")
r = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
    json={"model": "gpt-4o",
          "messages": [{"role": "user", "content": "Reply with only the word PING"}],
          "max_tokens": 5, "temperature": 0.0},
    timeout=20
)
print(f"Status: {r.status_code}")
print(f"Response: {r.text[:200]}\n")

if r.status_code != 200:
    print(">>> STOP. Fix OpenAI access before proceeding. <<<")
else:
    print(">>> Test passed. Running classification loop... <<<\n")

    def get_answer_gpt4o(question, options, api_key):
        prompt = f"""Answer this medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
        for attempt in range(3):
            try:
                r = requests.post(
                    "https://api.openai.com/v1/chat/completions",
                    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                    json={"model": "gpt-4o",
                          "messages": [{"role": "user", "content": prompt}],
                          "max_tokens": 5, "temperature": 0.0},
                    timeout=30
                )
                if r.status_code == 429:
                    time.sleep(5 * (attempt + 1))
                    continue
                if r.status_code != 200:
                    return None, f"HTTP {r.status_code}: {r.text[:150]}"
                text = r.json()['choices'][0]['message']['content'].strip().upper()
                for char in text:
                    if char in 'ABCD':
                        return char, None
                return None, f"No letter: {text!r}"
            except Exception as e:
                if attempt == 2:
                    return None, str(e)
                time.sleep(3)
        return None, "Max retries hit"

    results = []
    consecutive_errors = 0

    for i, q in enumerate(questions):
        pred, err = get_answer_gpt4o(q['question'], q['options'], OPENAI_API_KEY)

        if pred is None:
            consecutive_errors += 1
            if consecutive_errors <= 3:
                print(f"  [{i}] error: {err}")
            if consecutive_errors >= 10:
                print(f"  >>> 10 consecutive errors, aborting <<<")
                break
        else:
            consecutive_errors = 0

        results.append({
            'idx': q['idx'],
            'id': q['id'],
            'gold': q['answer_idx'],
            'pred': pred,
            'correct': int(pred == q['answer_idx']) if pred else 0,
        })

        if (i + 1) % 200 == 0:
            acc = sum(r['correct'] for r in results) / len(results) * 100

            with open(f'{PROJECT_DIR}/medmcqa_gpt4o_results_partial.json', 'w') as f:
                json.dump(results, f)
            print(f"  {i+1}/{len(questions)} — running acc: {acc:.1f}%  (checkpoint saved)")
        time.sleep(0.1)

    total = len(results)
    correct = sum(r['correct'] for r in results)
    print(f"\n=== GPT-4o on MedMCQA ===")
    print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
    print(f"  (DeepSeek-V3 on MedQA was 77.7% for reference)")

    pred_dist = Counter(r['pred'] for r in results)
    print(f"  Prediction distribution: {dict(pred_dist)}")

    with open(f'{PROJECT_DIR}/medmcqa_gpt4o_results.json', 'w') as f:
        json.dump(results, f)
    print(f"  Saved to {PROJECT_DIR}/medmcqa_gpt4o_results.json")